In [1]:
from datasets import load_dataset

dst = load_dataset("sh0416/ag_news")

C:\Users\79635\PycharmProjects\word2vec_scipgram_ns\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
len(dst["train"])

120000

In [3]:
dst["train"]["description"][100000]

" CARACAS, Venezuela (Reuters) - A Venezuelan lawyer  suspected in last week's bombing murder of a top state  prosecutor was killed in a gunfight with police on Tuesday  after he tried to ram detectives with his car and opened fire  on them, officials said."

In [4]:
dst["train"]["title"][100000]

'Venezuelan Car-Bomb Suspect Killed, Weapons Found'

In [5]:
from tqdm import tqdm

In [29]:
from preprocessing import *

word_counts = Counter()

for i in tqdm(range(12000)):
    example = dst["train"]["title"][i] + "." + dst["train"]["description"][i]
    tokenization(text=example, word_counts=word_counts)

100%|██████████| 12000/12000 [00:06<00:00, 1849.09it/s]


In [30]:
word_counts

Counter({'the': 20978,
         'to': 11973,
         'a': 11203,
         'of': 10235,
         'in': 10150,
         'and': 7048,
         'on': 5788,
         'for': 4995,
         'that': 2705,
         'with': 2636,
         'as': 2611,
         'at': 2569,
         'reuters': 2382,
         'by': 2272,
         'new': 2164,
         'is': 2125,
         'its': 2111,
         'said': 2017,
         'it': 1911,
         'from': 1910,
         'ap': 1893,
         'has': 1827,
         'an': 1772,
         'his': 1545,
         'after': 1476,
         'us': 1460,
         's': 1418,
         'was': 1376,
         'will': 1300,
         'u': 1178,
         'have': 1159,
         'up': 1116,
         'be': 1114,
         '39': 1101,
         'are': 1096,
         'over': 1092,
         'more': 1049,
         'oil': 1039,
         'their': 1035,
         'but': 1028,
         'first': 1024,
         'athens': 1004,
         'two': 989,
         'thursday': 928,
         'olympic': 920,

In [24]:
len(word_counts.keys())

76

In [31]:
word_counts.most_common(10)

[('the', 20978),
 ('to', 11973),
 ('a', 11203),
 ('of', 10235),
 ('in', 10150),
 ('and', 7048),
 ('on', 5788),
 ('for', 4995),
 ('that', 2705),
 ('with', 2636)]

In [32]:
unique_words = [
    w for w, c in word_counts.items()
    if c > 5
]

In [33]:
len(unique_words)

7672

In [34]:
word2id = {}
id2word = {}

for idx, word in enumerate(unique_words):
    word2id[word] = idx
    id2word[idx] = word

In [13]:
encode_text(text=dst["train"]["title"][100] + "." + dst["train"]["description"][100],
            word2id=word2id,)

[1492,
 1491,
 37,
 1498,
 907,
 31,
 1499,
 1500,
 1408,
 169,
 1408,
 169,
 31,
 1499,
 1500,
 660,
 65,
 1501,
 1492,
 37,
 1491,
 896,
 1502,
 65,
 320,
 1055,
 65,
 1498,
 1058,
 6,
 1503,
 1504,
 16,
 1505,
 355,
 1506,
 37,
 92,
 320,
 132,
 1052,
 132,
 1507,
 104,
 1508,
 1509]

In [35]:
def tokens_pairs(tokens: List[int], context_window: int, ignore_id: int = -1) -> List[Tuple[int, int]]:
    """
    Build (center, context) pairs for skip-gram.
    tokens: list of token ids for ONE document
    context_window: window size k
    ignore_id: token id to ignore (e.g. -1 for OOV)
    """
    pairs: List[Tuple[int, int]] = []
    n = len(tokens)

    for i, center in enumerate(tokens):
        if center == ignore_id:
            continue

        start = max(0, i - context_window)
        end = min(n, i + context_window + 1)

        for j in range(start, end):
            if j == i:
                continue

            context = tokens[j]
            if context == ignore_id:
                continue

            pairs.append((center, context))
    return pairs


In [37]:
from dataset import *

skipgram_pairs = []
context_window = 2

for i in tqdm(range(12000)):
    text = dst["train"]["title"][i] + "." + dst["train"]["description"][i]
    tokens = encode_text(text, word2id=word2id)
    pairs = tokens_pairs(tokens, context_window)
    skipgram_pairs.extend(pairs)

100%|██████████| 12000/12000 [00:07<00:00, 1658.97it/s]


In [38]:
len(dst["train"])

120000

In [39]:
len(skipgram_pairs)

1677204

In [40]:
#update word_counts
#because we deleted some words

word_counts = Counter({
    w: c for w, c in word_counts.items()
    if w in unique_words
})

In [41]:
len(word_counts) == len(unique_words)

True

In [42]:
import numpy as np

def build_unigram_probs(counts: np.ndarray, power: float = 0.75) -> np.ndarray:
    probs = counts ** power
    Z = probs.sum()
    probs /= Z
    return probs

In [43]:
word_counts_array = np.zeros(len(unique_words))

In [45]:
for i in range(len(word_counts_array)):
    current_word = id2word[i]
    freq = word_counts[current_word]
    word_counts_array[i] = freq

In [46]:
probs = build_unigram_probs(word_counts_array)

In [47]:
rng = np.random.default_rng(42)

In [48]:
from SkipGram_NegativeSampling import *
from training import *

model = Word2VecSGNS(vocab_size = len(unique_words), dim=100)
train(model=model, data=skipgram_pairs, num_epochs=5, k=5, batch_size=128,
      probs=probs, rng=rng)

 20%|██        | 1/5 [03:30<14:02, 210.65s/it]

epoch 1: loss=4.1584


 40%|████      | 2/5 [06:26<09:30, 190.30s/it]

epoch 2: loss=4.1358


 60%|██████    | 3/5 [10:18<06:58, 209.33s/it]

epoch 3: loss=3.9987


 80%|████████  | 4/5 [13:14<03:15, 195.92s/it]

epoch 4: loss=3.8060


100%|██████████| 5/5 [16:50<00:00, 202.02s/it]

epoch 5: loss=3.6469


In [55]:
from evaluation import *

get_top_n_similar_words(model, "olympic", word2id, id2word, n=10, use_central=False)

[('states', 0.9521808667790318),
 ('first', 0.949958043569739),
 ('united', 0.9494409709665159),
 ('us', 0.9465791973196673),
 ('athens', 0.9429316054440544),
 ('olympics', 0.9380519315554703),
 ('games', 0.9333965258637283),
 ('over', 0.9326959273790147),
 ("men's", 0.9269725820578071),
 ('at', 0.9230307235532831)]

In [75]:
get_top_n_similar_words(model, "one", word2id, id2word, n=5, use_central=True)

[('and', 0.9952791332479729),
 ('out', 0.9935305520599469),
 ('first', 0.9929795012585312),
 ('gold', 0.9922965773978034),
 ('time', 0.9919254583568801)]

In [80]:
get_top_n_similar_words(model, "one", word2id, id2word, n=10, use_central=False)

[('city', 0.9089027264841493),
 ('out', 0.892196064698633),
 ('us', 0.8742994011294063),
 ('first', 0.8724214885200388),
 ('two', 0.8721313873806382),
 ('olympic', 0.8616717816457939),
 ('most', 0.8605494296404621),
 ('second', 0.8596255746303454),
 ('athens', 0.858921670521158),
 ("men's", 0.8581890333073086)]